# 03b_import_mintaka_multihop

Ноутбук для **импорта Mintaka** и извлечения **multi-hop** запросов как следующего внешнего слоя для бенчмарка.

Что делает ноутбук:
- загружает raw `mintaka_train.json`, `mintaka_dev.json`, `mintaka_test.json` напрямую из GitHub-репозитория Mintaka;
- нормализует записи в единый формат;
- фильтрует только нужные типы сложности (по умолчанию `multihop`);
- экспортирует отобранный слой в `jsonl / json / csv`;
- сохраняет удобные summary-отчёты для ручного просмотра и для дальнейшей загрузки в инфру.

Важно:
- **Mintaka не содержит русского языка** — в датасете есть английский и 8 других языков, но не русский;
- поэтому этот слой рассматривается как **донор сложных паттернов и английских вопросов для дальнейшего перевода / рефраза в инфре**.

In [1]:

# =========================
# УСТАНОВКА / ИМПОРТЫ
# =========================
%pip install -q pandas requests matplotlib

import json
import os
import re
import hashlib
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd
import requests


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:

# =========================
# КОНФИГ
# =========================
PROJECT_ROOT = Path.cwd()
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts_mintaka_stage1"
CACHE_DIR = ARTIFACTS_DIR / "_cache_raw"
JSONL_DIR = ARTIFACTS_DIR / "jsonl"
CSV_DIR = ARTIFACTS_DIR / "csv"
REPORTS_DIR = ARTIFACTS_DIR / "reports"

for p in [ARTIFACTS_DIR, CACHE_DIR, JSONL_DIR, CSV_DIR, REPORTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# По умолчанию вытаскиваем только multi-hop.
# Если захочешь, можно расширить, например:
# INCLUDE_COMPLEXITY_TYPES = ["multihop", "intersection", "difference", "superlative", "ordinal", "comparative"]
INCLUDE_COMPLEXITY_TYPES = ["multihop"]

RAW_URLS = {
    "train": "https://raw.githubusercontent.com/amazon-science/mintaka/main/data/mintaka_train.json",
    "validation": "https://raw.githubusercontent.com/amazon-science/mintaka/main/data/mintaka_dev.json",
    "test": "https://raw.githubusercontent.com/amazon-science/mintaka/main/data/mintaka_test.json",
}

print("Artifacts dir:", ARTIFACTS_DIR.resolve())
print("Included complexity types:", INCLUDE_COMPLEXITY_TYPES)

Artifacts dir: /Users/matvey/Desktop/mintaka/artifacts_mintaka_stage1
Included complexity types: ['multihop']


In [3]:

# =========================
# БАЗОВЫЕ УТИЛИТЫ
# =========================
def normalize_spaces(text):
    if text is None:
        return ""
    text = str(text).replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def stable_hash(text, n=16):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:n]

def safe_get(d, key, default=None):
    if isinstance(d, dict):
        return d.get(key, default)
    return default

def ensure_list(x):
    if x is None:
        return []
    if isinstance(x, list):
        return x
    return [x]

def write_jsonl(rows, path):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

def write_json(rows, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)

def download_if_needed(url, out_path):
    out_path = Path(out_path)
    if out_path.exists():
        print(f"[cache] {out_path.name}")
        return out_path
    print(f"[download] {url}")
    resp = requests.get(url, timeout=120)
    resp.raise_for_status()
    out_path.write_bytes(resp.content)
    return out_path

In [4]:

# =========================
# ЗАГРУЗКА RAW JSON
# =========================
def load_raw_split(split_name):
    local_path = CACHE_DIR / f"mintaka_{split_name}.json"
    download_if_needed(RAW_URLS[split_name], local_path)
    with open(local_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    print(f"[loaded] split={split_name} rows={len(data):,}")
    return data

raw_train = load_raw_split("train")
raw_val = load_raw_split("validation")
raw_test = load_raw_split("test")

[download] https://raw.githubusercontent.com/amazon-science/mintaka/main/data/mintaka_train.json
[loaded] split=train rows=14,000
[download] https://raw.githubusercontent.com/amazon-science/mintaka/main/data/mintaka_dev.json
[loaded] split=validation rows=2,000
[download] https://raw.githubusercontent.com/amazon-science/mintaka/main/data/mintaka_test.json
[loaded] split=test rows=4,000


In [5]:

# =========================
# БЫСТРАЯ ПРОВЕРКА СТРУКТУРЫ
# =========================
example = raw_train[0]
print("Top-level keys:", list(example.keys()))
print()
print(json.dumps(
    {
        "id": example.get("id"),
        "question": example.get("question"),
        "category": example.get("category"),
        "complexityType": example.get("complexityType"),
        "answer_keys": list(example.get("answer", {}).keys()),
        "num_question_entities": len(example.get("questionEntity", [])),
    },
    ensure_ascii=False,
    indent=2
))

Top-level keys: ['id', 'question', 'translations', 'questionEntity', 'answer', 'category', 'complexityType']

{
  "id": "a9011ddf",
  "question": "What is the seventh tallest mountain in North America?",
  "category": "geography",
  "complexityType": "ordinal",
  "answer_keys": [
    "answerType",
    "answer",
    "mention"
  ],
  "num_question_entities": 2
}


In [6]:

# =========================
# СТАТИСТИКА ПО COMPLEXITY TYPE
# =========================
def complexity_stats(data, split_name):
    c = Counter([safe_get(x, "complexityType", "") for x in data])
    df = pd.DataFrame({
        "complexityType": list(c.keys()),
        "count": list(c.values()),
    }).sort_values(["count", "complexityType"], ascending=[False, True]).reset_index(drop=True)
    df["split"] = split_name
    return df[["split", "complexityType", "count"]]

stats_df = pd.concat([
    complexity_stats(raw_train, "train"),
    complexity_stats(raw_val, "validation"),
    complexity_stats(raw_test, "test"),
], ignore_index=True)

stats_df

,split,complexityType,count
0,train,generic,2800
1,train,comparative,1400
2,train,count,1400
3,train,difference,1400
4,train,intersection,1400
5,train,multihop,1400
6,train,ordinal,1400
7,train,superlative,1400
8,train,yesno,1400
9,validation,generic,400


In [7]:

# =========================
# НОРМАЛИЗАЦИЯ ОДНОЙ ЗАПИСИ
# =========================
def get_answer_mode(answer_obj):
    answer_type = safe_get(answer_obj, "answerType", "")
    answer_items = ensure_list(safe_get(answer_obj, "answer", []))

    if answer_type in {"boolean"}:
        return "boolean"
    if answer_type in {"number"}:
        return "count" if len(answer_items) <= 1 else "list_like"
    if answer_type in {"date", "string"}:
        return "single" if len(answer_items) <= 1 else "list_like"
    if answer_type in {"entity"}:
        return "single" if len(answer_items) <= 1 else "list_like"
    return "unknown"

def normalize_question_entities(question_entities):
    out = []
    for qe in ensure_list(question_entities):
        out.append({
            "name": str(safe_get(qe, "name", "")),
            "entity_type": safe_get(qe, "entityType", ""),
            "label": safe_get(qe, "label", ""),
            "mention": safe_get(qe, "mention", ""),
            "span": safe_get(qe, "span", []),
        })
    return out

def normalize_answer_entities(answer_obj):
    answer_items = ensure_list(safe_get(answer_obj, "answer", []))
    out = []
    for ae in answer_items:
        if isinstance(ae, dict):
            label = safe_get(ae, "label", "")
            if isinstance(label, dict):
                label_en = label.get("en", None)
            else:
                label_en = label
            out.append({
                "name": str(safe_get(ae, "name", "")),
                "label_en": label_en,
                "raw_label": label,
            })
        else:
            out.append({
                "name": str(ae),
                "label_en": None,
                "raw_label": None,
            })
    return out

def supporting_entities(answer_obj):
    out = []
    for ae in ensure_list(safe_get(answer_obj, "supportingEnt", [])):
        if isinstance(ae, dict):
            label = safe_get(ae, "label", "")
            if isinstance(label, dict):
                label_en = label.get("en", None)
            else:
                label_en = label
            out.append({
                "name": str(safe_get(ae, "name", "")),
                "label_en": label_en,
                "raw_label": label,
            })
        else:
            out.append({
                "name": str(ae),
                "label_en": None,
                "raw_label": None,
            })
    return out

def normalize_record(sample, split_name):
    answer_obj = safe_get(sample, "answer", {})
    answer_entities = normalize_answer_entities(answer_obj)
    support_ents = supporting_entities(answer_obj)
    support_nums = ensure_list(safe_get(answer_obj, "supportingNum", []))

    benchmark_id = f"mintaka_{split_name}_{safe_get(sample, 'id', stable_hash(json.dumps(sample, ensure_ascii=False)))}"
    answer_mode = get_answer_mode(answer_obj)

    gold_answers = []
    for ae in answer_entities:
        gold_answers.append({
            "name": ae["name"],
            "label_en": ae["label_en"],
        })

    gold_qids = [
        ae["name"] for ae in answer_entities
        if str(ae["name"]).startswith("Q")
    ]

    return {
        "benchmark_id": benchmark_id,
        "source_dataset": "mintaka",
        "source_split": split_name,
        "source_id": safe_get(sample, "id", ""),
        "question_en_original": normalize_spaces(safe_get(sample, "question", "")),
        "question_translations": safe_get(sample, "translations", {}),
        "category": safe_get(sample, "category", ""),
        "complexity_type": safe_get(sample, "complexityType", ""),
        "reasoning_family": safe_get(sample, "complexityType", ""),
        "question_entities": normalize_question_entities(safe_get(sample, "questionEntity", [])),
        "answer_type": safe_get(answer_obj, "answerType", ""),
        "answer_mode": answer_mode,
        "answer_text_en": normalize_spaces(safe_get(answer_obj, "mention", "")),
        "answer_entities": answer_entities,
        "gold_answers": gold_answers,
        "gold_qids": gold_qids,
        "supporting_entities": support_ents,
        "supporting_numbers": support_nums,
        "num_answers": len(answer_entities),
        "num_question_entities": len(safe_get(sample, "questionEntity", [])),
        "has_supporting_entities": len(support_ents) > 0,
        "has_supporting_numbers": len(support_nums) > 0,
        "needs_translation": True,
        "needs_rephrase": True,
        "translation_source": "en",
        "notes": "",
    }

In [8]:

# =========================
# НОРМАЛИЗАЦИЯ ВСЕХ SPLITS
# =========================
all_rows = []
for split_name, raw_data in [("train", raw_train), ("validation", raw_val), ("test", raw_test)]:
    split_rows = [normalize_record(x, split_name) for x in raw_data]
    print(f"[normalized] split={split_name} rows={len(split_rows):,}")
    all_rows.extend(split_rows)

all_df = pd.DataFrame(all_rows)
print("All normalized:", all_df.shape)
all_df.head(3)

[normalized] split=train rows=14,000
[normalized] split=validation rows=2,000
[normalized] split=test rows=4,000
All normalized: (20000, 26)


,benchmark_id,source_dataset,source_split,source_id,question_en_original,question_translations,category,complexity_type,reasoning_family,question_entities,...,supporting_entities,supporting_numbers,num_answers,num_question_entities,has_supporting_entities,has_supporting_numbers,needs_translation,needs_rephrase,translation_source,notes
0,mintaka_train_a9011ddf,mintaka,train,a9011ddf,What is the seventh tallest mountain in North ...,"{'ar': 'ما سابع أعلى جبل في أمريكا الشمالية؟',...",geography,ordinal,ordinal,"[{'name': 'Q49', 'entity_type': 'entity', 'lab...",...,[],[],1,2,False,False,True,True,en,
1,mintaka_train_2723bb1b,mintaka,train,2723bb1b,Which actor was the star of Titanic and was bo...,{'ar': 'مَنْ الممثل الذي لعب دور البطولة في في...,movies,intersection,intersection,"[{'name': 'Q44578', 'entity_type': 'entity', '...",...,[],[],1,2,False,False,True,True,en,
2,mintaka_train_88349c89,mintaka,train,88349c89,Which actor starred in Vanilla Sky and was mar...,{'ar': 'مَنْ الممثل الذي لعب دور البطولة في في...,movies,intersection,intersection,"[{'name': 'Q174346', 'entity_type': 'entity', ...",...,[],[],1,2,False,False,True,True,en,


In [9]:

# =========================
# ФИЛЬТР: НУЖНЫЕ COMPLEXITY TYPES
# =========================
filtered_df = all_df[all_df["complexity_type"].isin(INCLUDE_COMPLEXITY_TYPES)].copy()

print("Filtered shape:", filtered_df.shape)
print(filtered_df["complexity_type"].value_counts(dropna=False))
print()
print(filtered_df["source_split"].value_counts(dropna=False))

Filtered shape: (2000, 26)
complexity_type
multihop    2000
Name: count, dtype: int64

source_split
train         1400
test           400
validation     200
Name: count, dtype: int64


In [10]:

# =========================
# ДОПОЛНИТЕЛЬНАЯ ПОЛЕЗНАЯ СТАТИСТИКА
# =========================
print("Answer modes:")
print(filtered_df["answer_mode"].value_counts(dropna=False))
print()
print("Categories:")
print(filtered_df["category"].value_counts(dropna=False))
print()
print("Num answers distribution:")
print(filtered_df["num_answers"].value_counts(dropna=False).sort_index().head(20))

Answer modes:
answer_mode
single       1380
unknown       570
list_like      46
boolean         4
Name: count, dtype: int64

Categories:
category
music         250
movies        250
sports        250
books         250
geography     250
politics      250
videogames    250
history       250
Name: count, dtype: int64

Num answers distribution:
num_answers
0      52
1    1902
2      38
3       8
Name: count, dtype: int64


In [11]:

# =========================
# ПРИОРИТИЗАЦИЯ ДЛЯ ДАЛЬНЕЙШЕЙ РАБОТЫ
# =========================
def compute_priority_score(row):
    score = 0
    # multi-hop уже зафиксирован фильтром, но добавим небольшую базу
    score += 5

    # entity-answers особенно удобны как KB-паттерны
    if row["answer_type"] == "entity":
        score += 2

    # несколько answer entities потенциально интересны
    if row["num_answers"] > 1:
        score += 3

    # supporting entities/nums полезны для анализа
    if row["has_supporting_entities"]:
        score += 1
    if row["has_supporting_numbers"]:
        score += 1

    # слишком boolean-heavy вещи чуть понижаем
    if row["answer_mode"] == "boolean":
        score -= 2

    # count-like можно оставить, но чуть ниже приоритета
    if row["answer_mode"] == "count":
        score -= 1

    return score

filtered_df["priority_score"] = filtered_df.apply(compute_priority_score, axis=1)

filtered_df = filtered_df.sort_values(
    ["priority_score", "num_answers", "source_split", "benchmark_id"],
    ascending=[False, False, True, True]
).reset_index(drop=True)

filtered_df[[
    "benchmark_id", "question_en_original", "category", "complexity_type",
    "answer_mode", "num_answers", "priority_score"
]].head(20)

,benchmark_id,question_en_original,category,complexity_type,answer_mode,num_answers,priority_score
0,mintaka_test_4af81c8c,Which awards was the latest Animal Crossing ga...,videogames,multihop,list_like,3,10
1,mintaka_test_63690199,Who produced the movie that's based on the Har...,books,multihop,list_like,3,10
2,mintaka_train_11dfec2b,Where was the oldest member of The Rolling Sto...,music,multihop,list_like,3,10
3,mintaka_train_29e5bf22,Who were the playable characters of the fourth...,videogames,multihop,list_like,3,10
4,mintaka_train_34409836,Where was the former member of One Direction b...,music,multihop,list_like,3,10
5,mintaka_train_724db5c0,Where was the singer of Back in Black born?,music,multihop,list_like,3,10
6,mintaka_train_91c7335e,Who are the love interests in Mass Effect 1?,videogames,multihop,list_like,3,10
7,mintaka_train_b5103288,Where was Los Angeles Angels' two-way star born?,sports,multihop,list_like,3,10
8,mintaka_test_031c5618,Where is the lead singer of The Backstreet Boy...,music,multihop,list_like,2,10
9,mintaka_test_4f65e7e6,Where was the captain of the US women's nation...,sports,multihop,list_like,2,10


In [12]:

# =========================
# EXPORT
# =========================
all_rows_out = all_df.to_dict(orient="records")
filtered_rows_out = filtered_df.to_dict(orient="records")

write_jsonl(all_rows_out, JSONL_DIR / "mintaka_all_normalized.jsonl")
write_json(all_rows_out, JSONL_DIR / "mintaka_all_normalized.json")
all_df.to_csv(CSV_DIR / "mintaka_all_normalized.csv", index=False, encoding="utf-8")

write_jsonl(filtered_rows_out, JSONL_DIR / "mintaka_multihop_candidates.jsonl")
write_json(filtered_rows_out, JSONL_DIR / "mintaka_multihop_candidates.json")
filtered_df.to_csv(CSV_DIR / "mintaka_multihop_candidates.csv", index=False, encoding="utf-8")

# Минимальный плоский слой для загрузки в translation/rephrase pipeline
minimal_cols = [
    "benchmark_id",
    "source_dataset",
    "source_split",
    "source_id",
    "question_en_original",
    "category",
    "complexity_type",
    "reasoning_family",
    "answer_type",
    "answer_mode",
    "answer_text_en",
    "num_answers",
    "priority_score",
    "needs_translation",
    "needs_rephrase",
]
minimal_df = filtered_df[minimal_cols].copy()

write_jsonl(minimal_df.to_dict(orient="records"), JSONL_DIR / "mintaka_multihop_translation_minimal.jsonl")
write_json(minimal_df.to_dict(orient="records"), JSONL_DIR / "mintaka_multihop_translation_minimal.json")
minimal_df.to_csv(CSV_DIR / "mintaka_multihop_translation_minimal.csv", index=False, encoding="utf-8")

summary = {
    "all_rows": int(len(all_df)),
    "filtered_rows": int(len(filtered_df)),
    "included_complexity_types": INCLUDE_COMPLEXITY_TYPES,
    "answer_mode_counts": filtered_df["answer_mode"].value_counts(dropna=False).to_dict(),
    "category_counts": filtered_df["category"].value_counts(dropna=False).to_dict(),
    "split_counts": filtered_df["source_split"].value_counts(dropna=False).to_dict(),
}

with open(REPORTS_DIR / "mintaka_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("[saved] all normalized + filtered + minimal exports")
print("JSONL dir:", JSONL_DIR.resolve())
print("CSV dir:", CSV_DIR.resolve())
print("Reports dir:", REPORTS_DIR.resolve())

[saved] all normalized + filtered + minimal exports
JSONL dir: /Users/matvey/Desktop/mintaka/artifacts_mintaka_stage1/jsonl
CSV dir: /Users/matvey/Desktop/mintaka/artifacts_mintaka_stage1/csv
Reports dir: /Users/matvey/Desktop/mintaka/artifacts_mintaka_stage1/reports


In [13]:

# =========================
# PREVIEW ДЛЯ РУЧНОГО ПРОСМОТРА
# =========================
preview_cols = [
    "benchmark_id",
    "question_en_original",
    "category",
    "answer_text_en",
    "answer_type",
    "answer_mode",
    "num_answers",
    "priority_score",
]
filtered_df[preview_cols].head(50)

,benchmark_id,question_en_original,category,answer_text_en,answer_type,answer_mode,num_answers,priority_score
0,mintaka_test_4af81c8c,Which awards was the latest Animal Crossing ga...,videogames,"Best Family Game, Best Multiplayer Game, Game ...",entity,list_like,3,10
1,mintaka_test_63690199,Who produced the movie that's based on the Har...,books,"Chris Columbus, David Heyman, Mark Radcliffe",entity,list_like,3,10
2,mintaka_train_11dfec2b,Where was the oldest member of The Rolling Sto...,music,"Lewisham, London, England",entity,list_like,3,10
3,mintaka_train_29e5bf22,Who were the playable characters of the fourth...,videogames,"Niko Bellic, Johnny Klebitz, Luis Fernando Lopez",entity,list_like,3,10
4,mintaka_train_34409836,Where was the former member of One Direction b...,music,"Bradford, West Yorkshire, England",entity,list_like,3,10
5,mintaka_train_724db5c0,Where was the singer of Back in Black born?,music,"Dunston, County Durham, England",entity,list_like,3,10
6,mintaka_train_91c7335e,Who are the love interests in Mass Effect 1?,videogames,"Kaidan Alenko, Ashley Williams, Liara T'Soni",entity,list_like,3,10
7,mintaka_train_b5103288,Where was Los Angeles Angels' two-way star born?,sports,"Oshu, Iwate, Japan",entity,list_like,3,10
8,mintaka_test_031c5618,Where is the lead singer of The Backstreet Boy...,music,"Lexington, Kentucky",entity,list_like,2,10
9,mintaka_test_4f65e7e6,Where was the captain of the US women's nation...,sports,"Danvers, Massachusetts.",entity,list_like,2,10


In [14]:

# =========================
# ОПЦИОНАЛЬНО: CSV -> JSON КОНВЕРТЕР
# =========================
def csv_to_json(csv_path, json_path):
    df = pd.read_csv(csv_path)
    rows = df.to_dict(orient="records")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)
    print(f"[ok] {csv_path} -> {json_path} ({len(rows):,} rows)")

# Пример:
# csv_to_json(
#     CSV_DIR / "mintaka_multihop_translation_minimal.csv",
#     JSONL_DIR / "mintaka_multihop_translation_minimal_from_csv.json"
# )

## Что смотреть после запуска

В первую очередь:
- `artifacts_mintaka_stage1/jsonl/mintaka_multihop_candidates.jsonl`
- `artifacts_mintaka_stage1/csv/mintaka_multihop_candidates.csv`
- `artifacts_mintaka_stage1/jsonl/mintaka_multihop_translation_minimal.json`

Именно эти файлы удобнее всего использовать как следующий внешний слой для:
- ручного отбора;
- подготовки к переводу;
- загрузки в translation / rephrase pipeline в инфре.